# **Advanced Text Mining**

This session explores advanced text mining methodologies, with a particular focus on data collection and longitudinal analysis.

## Data Collection

There are many ways to collect data, especially digital text data.

As seen from our previous sessions, we already implement some ways of collecting data:
- Downloading data from Project Gutenberg using `requests` library to download the text directly from the website.
- Using `datasets` library to load data from Hugging Face's datasets repository.

Aside from those methods, there are also other ways to collect data, such as:
- Web scraping using libraries like `BeautifulSoup` or `Scrapy` to extract text data from websites.
- Using APIs provided by platforms like Twitter, Reddit, or news websites to collect text data.
- Collecting data from online forums, blogs, or social media platforms using tools like `Selenium` or `Puppeteer` for automated browsing and data extraction. This also one form of web scraping.
- Using online repositories like CLARIN platform to access linguistic data and resources.

For example, we will use `BeautifulSoup` to scrape abstracts books data from *Books to Scrape* website, which is a website that provides a collection of books for web scraping practice.


In [1]:
!pip install beautifulsoup4

In [2]:
from bs4 import BeautifulSoup
import requests

URL = "https://books.toscrape.com/"
response = requests.get(URL)
soup = BeautifulSoup(response.content, "html.parser")
books = soup.find_all("article", class_="product_pod")

In [3]:
import pandas as pd

books_metadata = []
for book in books:
    title = book.h3.a["title"]

    image_url = book.find("div", class_="image_container").a.img["src"]

    price = book.find("div", class_="product_price").p.text

    # Format for rating is "star-rating <rating>", where <rating> can be One, Two, Three, Four, or Five
    rating = book.p["class"][1]

    books_metadata.append(
        {
            "title": title,
            "price": price,
            "image_url": image_url,
            "rating": rating,
        }
    )


books_df = pd.DataFrame(books_metadata)
books_df.head()

,title,price,image_url,rating
0,A Light in the Attic,£51.77,media/cache/2c/da/2cdad67c44b002e7ead0cc35693c...,Three
1,Tipping the Velvet,£53.74,media/cache/26/0c/260c6ae16bce31c8f8c95daddd9f...,One
2,Soumission,£50.10,media/cache/3e/ef/3eef99c9d9adef34639f51066202...,One
3,Sharp Objects,£47.82,media/cache/32/51/3251cf3a3412f53f339e42cac213...,Four
4,Sapiens: A Brief History of Humankind,£54.23,media/cache/be/a5/bea5697f2534a2f86a3ef27b5a8c...,Five


Let's cleanup by appending the base url to the `image_url`, change the `rating` to be numeric instead of string, split the `price` column into `currency` and `price`, and convert the `price` column to numeric.

In [4]:
books_df["image_url"] = books_df["image_url"].apply(lambda x: URL + x)
books_df["rating"] = books_df["rating"].apply(
    lambda x: {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}[x]
)
books_df[["currency", "price"]] = books_df["price"].str.extract(r"([£$])(\d+\.\d{2})")
books_df["price"] = books_df["price"].astype(float)

books_df.head()

,title,price,image_url,rating,currency
0,A Light in the Attic,51.77,https://books.toscrape.com/media/cache/2c/da/2...,3,£
1,Tipping the Velvet,53.74,https://books.toscrape.com/media/cache/26/0c/2...,1,£
2,Soumission,50.10,https://books.toscrape.com/media/cache/3e/ef/3...,1,£
3,Sharp Objects,47.82,https://books.toscrape.com/media/cache/32/51/3...,4,£
4,Sapiens: A Brief History of Humankind,54.23,https://books.toscrape.com/media/cache/be/a5/b...,5,£


Although we cleaned up the data after we collected it, we can also do some cleaning during the data collection process itself, either way is fine as long as we end up with clean data that we can use for further processing or analysis.

### Specialized Libraries

Other ways to collect data is by using specialized libraries that are designed for specific types of data or platforms.

Here we will use the library [`newspaper4k`](https://github.com/AndyTheFactory/newspaper4k) to collect news articles data from the web. This library provides an easy-to-use interface for scraping and parsing news articles from various sources.

In [5]:
!pip install newspaper4k[nlp]

In [6]:
import newspaper

news_url = "https://www.bbc.com/news/articles/cq8ww7d72wyo"

article = newspaper.article(news_url)

print("Author:", article.authors)

print("Title:", article.title)

print("Publish Date:", article.publish_date)

print("Text:", article.text)

print("Top Image:", article.top_image)

print("Movies:", article.movies)

print("Summary:", article.summary)

Author: ['Ian Aikman']
Title: Man wins €1m Picasso painting in €100 charity raffle
Publish Date: 2026-04-14 21:40:24.923000+00:00
Text: Ari Hodara, an engineer and art enthusiast, learned he was the winner on Tuesday when he answered a video call from Christie's auction house in Paris.

"How do I know this isn't a prank?" the 58-year-old asked when he was told he was the new owner of the 1941 work by the Spanish master.

Organisers said more than 120,000 tickets for the prize draw were sold at €100 (£87; $118) each, raising around €11m (£10m; $13m) for Alzheimer's research.

The draw was the third edition of the "1 Picasso for 100 euros" fundraising raffle, which was founded in 2013.

This year's prize was Tête de Femme (Head of a Woman), a gouache-on-paper portrait rendered in Picasso's signature style. It depicts his partner and muse, the French surrealist artist Dora Maar.

"I was surprised, that's it," said Hodara during a phone call with auctioneers after the draw. "When you bet o

In [8]:
bbc_paper = newspaper.build("https://www.bbc.com")
print("Categories: ", bbc_paper.category_urls())

article_urls = [article.url for article in bbc_paper.articles]
print("Latest 3 articles: ", article_urls[:3])

if bbc_paper.articles:
    article = bbc_paper.articles[0]
    article.download()
    article.parse()

    print("Title:", article.title)

Categories:  ['https://www.bbc.com/igbo', 'https://www.bbc.com/live', 'https://www.bbc.com/hindi', 'https://www.bbc.com/hausa', 'https://www.bbc.com/gahuza', 'https://www.bbc.com/tamil', 'https://www.bbc.com/kyrgyz', 'https://www.bbc.com/bengali', 'https://www.bbc.com/sinhala', 'https://www.bbc.com/sport', 'https://www.bbc.com/audio', 'https://www.bbc.com/tigrinya', 'https://www.bbc.com/amharic', 'https://www.bbc.co.uk/accessibility', 'https://www.bbc.com/future-planet', 'https://www.bbc.com/azeri', 'https://www.bbc.com/marathi', 'https://www.bbc.com/korean', 'https://www.bbc.com/travel', 'https://www.bbc.com/video', 'https://www.bbc.com/home', 'https://www.bbc.com/vietnamese', 'https://www.bbc.com/russian', 'https://www.bbc.com/business', 'https://www.bbc.com/mundo', 'https://www.bbc.com/punjabi', 'https://www.bbc.com/burmese', 'https://www.bbc.com/pidgin', 'https://www.bbc.com/indonesia', 'https://www.bbc.com/somali', 'https://www.bbc.com/thai', 'https://www.bbc.com/yoruba', 'https:/

For more advanced uses of scraping, using tools like `Selenium` can be useful to automate browsing and data extraction from websites that require interaction or have dynamic content. This can be particularly useful for collecting data from online forums, blogs, or social media platforms where the content is generated dynamically based on user interactions.

For further reading on web scraping, you can check out the following resources:
- [Web Scraping with BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Web Scraping with Scrapy](https://docs.scrapy.org/en/latest/)
- [Web Scraping with Selenium](https://www.selenium.dev/documentation/webdriver/getting_started/)

## Dynamic Topic Modeling

_Adapted from BERTopic's documentation on [dynamic topic modeling](https://maartengr.github.io/BERTopic/getting_started/topicsovertime/topicsovertime.html)._

Dynamic Topic Modeling is a technique used to analyze the evolution of topics over time in a collection of documents. It allows us to identify how topics emerge, evolve, and fade away over time, providing insights into the changing themes and trends in a corpus of text data. This can be particularly useful for analyzing news articles, social media posts, or any other type of text data that is generated over time.

By using BERTopic, we can easily calculate the topic representation at each timestep and analyze how topics evolve over time, providing valuable insights into the changing themes and trends in our text data.

The main difference with our previous session is that for this we need temporal information in our data, which can be in the form of timestamps or any other temporal markers that indicate when each document was created or published. This allows us to track the evolution of topics over time and identify trends and patterns in our text data.

In this example we will use topic modeling to analyze the Tweet data of Donald Trump.

In [12]:
!pip install bertopic
!pip install "pandas<3.0"

# Suppress transformers logging
from transformers import logging

logging.set_verbosity_error()

In [13]:
import pandas as pd
import re
from functools import cache

@cache
def get_trump():
    trump_tweets_url = (
        "https://drive.google.com/uc?export=download&id=1xRKHaP-QwACMydlDnyFPEaFdtskJuBa6"
    )

    # Load the dataset
    trump = pd.read_csv(trump_tweets_url)

    # Preprocess the text data
    trump["text"] = (
        trump["text"]
        .apply(lambda x: re.sub(r"http\S+", "", x).strip().lower())
        .apply(lambda x: " ".join(filter(lambda x: x[0] != "@", x.split())))
        .apply(lambda x: " ".join(re.sub("[^a-zA-Z]+", " ", x).split()))
    )

    # Filter only original tweets (not retweets) and non-empty text
    return trump.query("isRetweet == 'f' and text != ''").reset_index(drop=True)

trump = get_trump()
trump.head()

,id,text,isRetweet,isDeleted,device,favorites,retweets,date,isFlagged
0,98454970654916608,republicans and democrats have both created ou...,f,f,TweetDeck,49,255,2011-08-02 18:07:48,f
1,1234653427789070336,i was thrilled to be back in the great city of...,f,f,Twitter for iPhone,73748,17404,2020-03-03 01:34:50,f
2,1304875170860015617,the unsolicited mail in ballot scam is a major...,f,f,Twitter for iPhone,80527,23502,2020-09-12 20:10:58,f
3,1223640662689689602,getting a little exercise this morning,f,f,Twitter for iPhone,285863,30209,2020-02-01 16:14:02,f
4,1215247978966986752,thank you elise,f,f,Twitter for iPhone,48510,11608,2020-01-09 12:24:31,f


In [14]:
timestamps = trump["date"].tolist()
tweets = trump["text"].tolist()

Then we can train the BERTopic model on the data and use the `topics_over_time` method to calculate the topic representation at each timestep. This will allow us to analyze how topics evolve over time and identify trends and patterns in the Tweet data.

The parameter `nr_bins` specifies the number of time bins to use for the analysis. This determines how the temporal information is grouped and can affect the granularity of the topic evolution analysis. By adjusting this parameter, we can control how detailed or coarse-grained our analysis of topic evolution over time will be.

In [15]:
from bertopic import BERTopic

model = BERTopic().fit(tweets)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [16]:
topics_over_time = model.topics_over_time(tweets, timestamps, nr_bins=20)
topics_over_time.head()

,Topic,Words,Frequency,Timestamp
0,-1,"donald, britney, igoogle, fb, urls",19,2009-04-30 12:30:07.597
1,0,"direct, randal, pinkett, precipice, lieutenant",9,2009-04-30 12:30:07.597
2,2,"begun, schedule, ahead, international, scotland",1,2009-04-30 12:30:07.597
3,9,"apprentice, celebrity, appearing, blog, lessons",2,2009-04-30 12:30:07.597
4,15,"execute, imagination, lot, entrepreneurs, donald",1,2009-04-30 12:30:07.597


From the obtained `topics_over_time` DataFrame, we can analyze the evolution of topics over time and identify trends and patterns in the Tweet data. We can visualize using the method `visualize_topics_over_time` to see how topics evolve over time and identify trends and patterns in the Tweet data. This can provide valuable insights into the changing themes and trends in the text data over time.

The parameter `top_n_topics` specifies the number of top topics to visualize in the plot. This allows us to focus on the most prominent topics and their evolution over time, providing a clearer view of the changing themes and trends in the text data. By adjusting this parameter, we can control how many topics are included in the visualization and gain insights into the most significant topics in our analysis.

In [18]:
model.visualize_topics_over_time(topics_over_time, top_n_topics=5)